Задание 2. Запрограммировать представления из заданий 1а-г (в виде одно- или многомерных массивов, словарей и т. д.). Можно использовать Python (numpy), C/C++.


In [14]:
import numpy as np

adj_matrix = np.zeros((5,5), dtype=int)

edges = [    
    (0, 1), # дуга 0
    (0, 3), # дуга 1
    (4, 1), # дуга 2
    (1, 3), # дуга 3
    (3, 2), # дуга 4
    (4, 3), # дуга 5
    (4, 0), # дуга 6
    (1, 2)  # дуга 7
]

for u, v in edges:
    adj_matrix[u][v] = 1

print("Матрица смежности:")
print(adj_matrix)

Матрица смежности:
[[0 1 0 1 0]
 [0 0 1 1 0]
 [0 0 0 0 0]
 [0 0 1 0 0]
 [1 1 0 1 0]]


In [16]:
inc_matrix = np.zeros((num_vertices, len(edges)), dtype=int)
for idx, (u, v) in enumerate(edges):
    inc_matrix[u][idx] = 1 # Начало
    inc_matrix[v][idx] = -1  # Конец

print("Матрица инцидентности:")
print(inc_matrix)

Матрица инцидентности:
[[ 1  1  0  0  0  0 -1  0]
 [-1  0 -1  1  0  0  0  1]
 [ 0  0  0  0 -1  0  0 -1]
 [ 0 -1  0 -1  1 -1  0  0]
 [ 0  0  1  0  0  1  1  0]]


In [17]:
adj_list = {i: [] for i in range(5)}
for u, v in edges:
    adj_list[u].append(v)

print("Список смежности:")
for node in sorted(adj_list.keys()):
    print(f"{node}: {adj_list[node]}")

Список смежности:
0: [1, 3]
1: [3, 2]
2: []
3: [2]
4: [1, 3, 0]


In [18]:
edge_list = edges.copy()
print("\nСписок дуг:", edge_list)


Список дуг: [(0, 1), (0, 3), (4, 1), (1, 3), (3, 2), (4, 3), (4, 0), (1, 2)]


In [22]:
sorted_edges = sorted(edges, key=lambda x: (x[0], x[1]))
offsets = [0] * (num_vertices + 1)
current_idx = 0
for v in range(num_vertices):
    offsets[v] = current_idx
    count = 0
    for u, w in sorted_edges:
        if u == v:
            count += 1
    current_idx += count
offsets[num_vertices] = len(sorted_edges) 
target_vertices = [v for u, v in sorted_edges]

print("Упорядоченное представление:")
print(f"Массив дуг: {sorted_edges}")
print(f"Массив смещений: {offsets}")

Упорядоченное представление:
Массив дуг: [(0, 1), (0, 3), (1, 2), (1, 3), (3, 2), (4, 0), (4, 1), (4, 3)]
Массив смещений: [0, 2, 4, 4, 5, 8]


Задание 3. Написать функцию для просмотра дуг, исходящих из данной вершины. В качестве аргументов: граф (может быть в различных представлениях), название представления графа (матрица смежности, матрица инцидентности, список смежности, список дуг, упорядоченный список дуг), номер вершины. На выходе: список кортежей вида (i, j), где i - вершина, из которой выходит дуга, j - вершина, в которую входит дуга).

In [26]:
def get_outgoing_edges(graph, rep_type, vertex, offsets=None):
    outgoing = []
    
    if rep_type == 'csr':
        if offsets is None:
            raise ValueError("Для представления 'csr' необходим аргумент offsets")
        
        start_idx = offsets[vertex]
        end_idx = offsets[vertex + 1]
        
        targets = graph[start_idx:end_idx]
        
        for target in targets:
            outgoing.append((vertex, target))
            
    elif rep_type == 'adj_matrix':
        n = len(graph)
        for j in range(n):
            if graph[vertex][j] == 1:
                outgoing.append((vertex, j))
                
    elif rep_type == 'inc_matrix':
        num_edges = graph.shape[1]
        for e_idx in range(num_edges):
            if graph[vertex][e_idx] == -1:
                for end_v in range(graph.shape[0]):
                    if graph[end_v][e_idx] == 1:
                        outgoing.append((vertex, end_v))
                        break
                        
    elif rep_type == 'adj_list':
        neighbors = graph.get(vertex, [])
        for neighbor in neighbors:
            outgoing.append((vertex, neighbor))
            
    elif rep_type in ['edge_list', 'ordered_edge_list']:
        for u, v in graph:
            if u == vertex:
                outgoing.append((u, v))
                
    else:
        raise ValueError(f"Неизвестный тип: {rep_type}")
        
    return outgoing

print("Тест")
# graph_csr_edges = target_vertices, offsets = offsets
out_csr = get_outgoing_edges(target_vertices, 'csr', 4, offsets=offsets)
print(f"Исходящие из 4: {out_csr}")

Тест
Исходящие из 4: [(4, 0), (4, 1), (4, 3)]


Задание 4. Написать функцию для перевода графа из одного представления в другое. Аргументы функции: граф, название начального представления графа, название требуемого представления графа (кроме упорядоченного списка дуг). Функция должна вернуть граф в требуемом представлении

In [28]:
def convert_graph(graph, from_rep, to_rep, offsets=None):
    temp_edge_list = []
    
    if from_rep == 'csr':
        if offsets is None:
            raise ValueError("Нужны offsets для разбора CSR")
        num_vertices = len(offsets) - 1
        for u in range(num_vertices):
            start = offsets[u]
            end = offsets[u+1]
            for idx in range(start, end):
                v = graph[idx] 
                temp_edge_list.append((u, v))
                
    elif from_rep == 'adj_matrix':
        n = len(graph)
        for i in range(n):
            for j in range(n):
                if graph[i][j] == 1:
                    temp_edge_list.append((i, j))
                    
    elif from_rep == 'inc_matrix':
        num_edges = graph.shape[1]
        for e_idx in range(num_edges):
            start = end = None
            for v in range(graph.shape[0]):
                if graph[v][e_idx] == -1: start = v
                elif graph[v][e_idx] == 1: end = v
            if start is not None and end is not None:
                temp_edge_list.append((start, end))
                
    elif from_rep == 'adj_list':
        for u in sorted(graph.keys()):
            for v in graph[u]:
                temp_edge_list.append((u, v))
                
    elif from_rep in ['edge_list', 'ordered_edge_list']:
        temp_edge_list = graph.copy()
        
    else:
        raise ValueError(f"Неподдерживаемый from_rep: {from_rep}")

    
    if to_rep == 'csr':
        sorted_edges = sorted(temp_edge_list, key=lambda x: (x[0], x[1]))
        
        if not sorted_edges:
            return ([], [0]*(num_vertices+1)) 
            
        max_v = max(max(u, v) for u, v in sorted_edges)
        num_v = max_v + 1
        
        new_offsets = [0] * (num_v + 1)
        new_targets = []
        
        current_idx = 0
        for v in range(num_v):
            new_offsets[v] = current_idx
            for u, w in sorted_edges:
                if u == v:
                    new_targets.append(w)
                    current_idx += 1
        new_offsets[num_v] = len(new_targets)
        
        return (new_targets, new_offsets)

    elif to_rep == 'adj_matrix':
        if not temp_edge_list:
             size = 5 
             return np.zeros((size, size), dtype=int)
             
        max_v = max(max(u, v) for u, v in temp_edge_list)
        size = max_v + 1
        new_adj = np.zeros((size, size), dtype=int)
        for u, v in temp_edge_list:
            new_adj[u][v] = 1
        return new_adj
        
    elif to_rep == 'inc_matrix':
        if not temp_edge_list:
            return np.array([]).reshape(0,0)
        max_v = max(max(u, v) for u, v in temp_edge_list)
        num_v = max_v + 1
        num_e = len(temp_edge_list)
        new_inc = np.zeros((num_v, num_e), dtype=int)
        for idx, (u, v) in enumerate(temp_edge_list):
            new_inc[u][idx] = -1
            new_inc[v][idx] = 1
        return new_inc
        
    elif to_rep == 'adj_list':
        if not temp_edge_list:
            return {}
        max_v = max(max(u, v) for u, v in temp_edge_list)
        new_adj_list = {i: [] for i in range(max_v + 1)}
        for u, v in temp_edge_list:
            new_adj_list[u].append(v)
        return new_adj_list
        
    elif to_rep == 'edge_list':
        return temp_edge_list
        
    elif to_rep == 'ordered_edge_list':
        return sorted(temp_edge_list, key=lambda x: (x[0], x[1]))
        
    else:
        raise ValueError(f"Неподдерживаемый to_rep: {to_rep}")

print("Тест конвертации в CSR")
csr_result = convert_graph(adj_list, 'adj_list', 'csr')
targets, offs = csr_result
print(f"CSR Targets: {targets}")
print(f"CSR Offsets: {offs}")

Тест конвертации в CSR
CSR Targets: [1, 3, 2, 3, 2, 0, 1, 3]
CSR Offsets: [0, 2, 4, 4, 5, 8]
